In [ ]:
import socket
import struct

A = 1.4
B = 0.3
x, y = 0.0, 0.0
HEADER = struct.Struct(">I")


def henon_map(x, y):
    return 1 - A * x**2 + y, B * x


def generate_chaos_sequence(length):
    """Advance the shared map state and return `length` keystream bytes."""
    global x, y
    seq = []
    for _ in range(length):
        x, y = henon_map(x, y)
        seq.append(int(x * 255) % 256)
    return seq


def encrypt_message(message):
    plaintext = message.encode("utf-8")
    keystream = generate_chaos_sequence(len(plaintext))
    return bytes(p ^ k for p, k in zip(plaintext, keystream))

In [ ]:
HOST = "127.0.0.1"
PORT = 65432

client_socket = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
client_socket.connect((HOST, PORT))

while True:
    message = input("Enter message to send (or 'exit'): ")
    if message.lower() == "exit":
        break
    encrypted = encrypt_message(message)
    client_socket.sendall(HEADER.pack(len(encrypted)) + encrypted)

client_socket.close()